In [48]:
import xarray as xr
import pandas as pd
import numpy as np
import cartopy.crs as ccrs
from cartopy.util import add_cyclic_point
import cartopy.feature as cf
import matplotlib.pyplot as plt
import matplotlib
from pathlib import Path
import matplotlib.ticker as ticker
%run utils/utils_helper_funcs.ipynb

Exception: File `'utils/utils_helper_funcs.ipynb'` not found.

In [12]:
def map_seasonal_sfc_ozone(ds, season, time_range=None):
    # Handle dates
    if time_range != None:
        start = str(time_range[0])
        end = str(time_range[1])
        ds = ds.sel(time=slice(start, end))
    else:
        start = str((ds['time'].values)[0])[:4]
        end = str((ds['time'].values)[len(ds['time'])-1])[:4]

    # Handle seasons
    if str(season)=='winter':
         s = 'DJF'
    elif str(season)=='spring':
        s = 'MAM'
    elif str(season)=='summer':
        s = 'JJA'
    elif str(season)=='fall':
        s = 'SON'
    else:
        raise ValueError('Please choose a correct season from the list: ["winter", "spring", "summer", "fall"]')

    ds_seasonal = ds.groupby('time.season').mean(skipna=True)
    layer_ozone = ds_seasonal.O3.sel(season = s, lev=max(ds_seasonal.lev.values))

    # Change longitude for plotting
    if max(ds_seasonal.lon)>350:
        layer_ozone = layer_ozone.assign_coords(lon=(((layer_ozone.lon + 180) % 360) - 180)).sortby('lon')
    layer_ozone = layer_ozone * 1e9

    # Create the plot
    proj = ccrs.PlateCarree()
    fig = plt.figure(figsize=(10,12))
    ax = plt.axes(projection = proj)
    ax.coastlines()
    ax.add_feature(cf.BORDERS.with_scale('50m'), edgecolor = [.3,.3,.3], linewidth = 0.5)

    ozone_cyc, lon_cyc = add_cyclic_point(layer_ozone, coord = layer_ozone.lon)
    o = ax.contourf(lon_cyc, layer_ozone.lat, ozone_cyc, transform=ccrs.PlateCarree(), cmap=matplotlib.cm.YlOrRd, levels = np.linspace(0,90.00, 19), extend='max')
    cb = plt.colorbar(o, extend = 'max', extendfrac = 'auto', shrink=0.25)
    cb_format = ticker.FuncFormatter(lambda x, pos: f'{x*1:.1f}')
    
    s_title=season.capitalize()
    cb.set_label('Ozone Concentration (ppb)')
    cb.ax.yaxis.set_major_formatter(cb_format)
    plt.title(f'Mean {s_title} Surface Ozone Concentrations, {start} - {end}')

In [7]:
def map_seasonal_trop_ozone(ds, season, time_range=None):        
    # Handle dates
    if time_range != None:
        start = str(time_range[0])
        end = str(time_range[1])
        ds = ds.sel(time=slice(start, end))
    else:
        start = str((ds['time'].values)[0])[:4]
        end = str((ds['time'].values)[len(ds['time'])-1])[:4]

    # Handle seasons
    if str(season)=='winter':
        s = 'DJF'
    elif str(season)=='spring':
        s = 'MAM'
    elif str(season)=='summer':
        s = 'JJA'
    elif str(season)=='fall':
        s = 'SON'
    else:
        raise ValueError('Please choose a correct season from the list: ["winter", "spring", "summer", "fall"]')

    ds.groupby('time.season').mean(skipna=True)
    inspect(ds_seasonal)
    p0 = ds_seasonal.P0 / 100 # Convert to hPa
    seasonal_ps = ds_seasonal.PS / 100    
    ozone = ds_seasonal.O3
    
    # Handle pressure
    pres_top = xr.zeros_like(ozone)
    pres_top = ds_seasonal.hyai*p0+ds_seasonal.hybi*seasonal_ps
    threshold = 100
    
    pressure = pres_top.where(pres_top >= threshold, drop=False) # Working with troposphere
    pres_arr = pressure.isel(ilev=slice(0,-1), lat=0, lon=0, time=0).drop_vars(['ilev','time','lat','lon'])
    layer_ozone = ozone.assign_coords(lev=pres_arr.data)
    layer_ozone = layer_ozone.where(~np.isnan(layer_ozone.lev), drop=True)
    
    
    # Change longitude for plotting
    if max(ds_seasonal.lon)>350:
        layer_ozone = layer_ozone.assign_coords(lon=(((layer_ozone.lon + 180) % 360) - 180)).sortby('lon')
    layer_ozone = layer_ozone.sel(season=s).mean()

    # Create the plot
    proj = ccrs.PlateCarree()
    fig = plt.figure(figsize=(10,12))
    ax = plt.axes(projection = proj)
    ax.coastlines()
    ax.add_feature(cf.BORDERS.with_scale('50m'), edgecolor = [.3,.3,.3], linewidth = 0.5)

    ozone_cyc, lon_cyc = add_cyclic_point(layer_ozone, coord = layer_ozone.lon)
    o = ax.contourf(lon_cyc, layer_ozone.lat, ozone_cyc, transform=ccrs.PlateCarree(), cmap=matplotlib.cm.YlOrRd, extend='max')
    cb = plt.colorbar(o, extend='max', extendfrac = 'auto', shrink=0.25)
    cb_format = ticker.FuncFormatter(lambda x, pos: f'{x*1e9:.1f}')
    
    s_title = season.capitalize()
    cb.set_label('Ozone Concentration (ppb)')
    cb.ax.yaxis.set_major_formatter(cb_format)
    plt.title(f'Mean {s_title} Tropospheric Ozone Concentrations, {start} - {end}')

In [8]:
def map_seasonal_strato_ozone(ds, season, time_range=None):
    # Handle dates
    if time_range != None:
        start = str(time_range[0])
        end = str(time_range[1])
        ds = ds.sel(time=slice(start, end))
    else:
        start = str((ds['time'].values)[0])[:4]
        end = str((ds['time'].values)[len(ds['time'])-1])[:4]

    # Handle seasons
    if str(season)=='winter':
         s = 'DJF'
    elif str(season)=='spring':
        s = 'MAM'
    elif str(season)=='summer':
        s = 'JJA'
    elif str(season)=='fall':
        s = 'SON'
    else:
        raise ValueError('Please choose a correct season from the list: ["winter", "spring", "summer", "fall"]')
            
    ds_seasonal = ds.groupby('time.season').mean(skipna=True)
    inspect(ds_seasonal)
    p0 = ds_seasonal.P0 / 100 # Convert to hPa
    seasonal_ps = ds_seasonal.PS / 100    
    ozone = ds_seasonal.O3
    
    # Handle pressure
    pres_bottom = xr.zeros_like(ozone)
    pres_bottom = ds_seasonal.hyai.shift(ilev=-1)*p0+ds_seasonal.hybi.shift(ilev=-1)*seasonal_ps
    threshold = 100
    
    pressure = pres_bottom.where(pres_bottom < threshold, drop=False)
    pres_arr = pressure.isel(ilev=slice(0,-1), lat=0, lon=0, time=0).drop_vars(['ilev','time','lat','lon'])
    layer_ozone = ozone.assign_coords(lev=pres_arr.data)
    layer_ozone = layer_ozone.where(~np.isnan(layer_ozone.lev), drop=True)

    # Change longitude for plotting
    if max(ds_seasonal.lon)>350:
        layer_ozone = layer_ozone.assign_coords(lon=(((layer_ozone.lon + 180) % 360) - 180)).sortby('lon')
    layer_ozone = layer_ozone.sel(season=s).mean()

    # Create the plot
    proj = ccrs.PlateCarree()
    fig = plt.figure(figsize=(10,12))
    ax = plt.axes(projection = proj)
    ax.coastlines()
    ax.add_feature(cf.BORDERS.with_scale('50m'), edgecolor = [.3,.3,.3], linewidth = 0.5)

    ozone_cyc, lon_cyc = add_cyclic_point(layer_ozone, coord = layer_ozone.lon)
    o = ax.contourf(lon_cyc, layer_ozone.lat, ozone_cyc, transform=ccrs.PlateCarree(), cmap=matplotlib.cm.YlOrRd, extend='max')
    cb = plt.colorbar(o, extend='max', extendfrac = 'auto', shrink=0.25)
    cb_format = ticker.FuncFormatter(lambda x, pos: f'{x*1e9:.1f}')
    
    s_title=season.capitalize()
    cb.set_label('Ozone Concentration (ppb)')
    cb.ax.yaxis.set_major_formatter(cb_format)
    plt.title(f'Mean {s_title} Stratospheric Ozone Concentrations, {start} - {end}')

In [2]:
def map_seasonal_trop_pm25(ds, season, time_range=None):
    # Handle dates
    if time_range != None:
        start = str(time_range[0])
        end = str(time_range[1])
        ds = ds.sel(time=slice(start, end))
    else:
        start = str((ds['time'].values)[0])[:4]
        end = str((ds['time'].values)[len(ds['time'])-1])[:4]

    # Handle seasons
    if str(season)=='winter':
         s = 'DJF'
    elif str(season)=='spring':
        s = 'MAM'
    elif str(season)=='summer':
        s = 'JJA'
    elif str(season)=='fall':
        s = 'SON'
    else:
        raise ValueError('Please choose a correct season from the list: ["winter", "spring", "summer", "fall"]')
            
    seasons = xr.groupers.SeasonResampler([s], drop_incomplete=False)
    ds_seasonal = ds.resample({'time':seasons}).mean() 
    inspect(ds_seasonal)
    p0 = ds_seasonal.P0 / 100 # Convert to hPa
    seasonal_ps = ds_seasonal.PS / 100    
    
    try:
        pm25 = ds_seasonal.PM25
    except:
        raise AttributeError('PM25 is not an attribute of this dataset')

    # Convert from kg/m^3 to ppb
    pm25 = (pm25*1e9)/0.66
    
    # Handle pressure
    pres_top = xr.zeros_like(pm25)
    pres_top = ds_seasonal.hyai*p0+ds_seasonal.hybi*seasonal_ps
    threshold = 100
    
    pressure = pres_top.where(pres_top >= threshold, drop=False) # Working with troposphere
    pres_arr = pressure.isel(ilev=slice(0,-1), lat=0, lon=0, time=0).drop_vars(['ilev','time','lat','lon'])
    layer_pm25 = pm25.assign_coords(lev=pres_arr.data)
    layer_pm25 = layer_pm25.where(~np.isnan(layer_pm25.lev), drop=True)
    
    # Change longitude for plotting
    if max(ds_seasonal.lon)>350:
        layer_pm25 = layer_pm25.assign_coords(lon=(((layer_pm25.lon + 180) % 360) - 180)).sortby('lon')
    layer_pm25 = layer_pm25.mean(dim=['lev','time'])

    # Create the plot
    proj = ccrs.PlateCarree()
    fig = plt.figure(figsize=(10,12))
    ax = plt.axes(projection = proj)
    ax.coastlines()
    ax.add_feature(cf.BORDERS.with_scale('50m'), edgecolor = [.3,.3,.3], linewidth = 0.5)

    data_cyc, lon_cyc = add_cyclic_point(layer_pm25, coord = layer_pm25.lon)
    o = ax.contourf(lon_cyc, layer_pm25.lat, data_cyc, transform=ccrs.PlateCarree(), cmap=matplotlib.cm.RdPu, extend='max')
    cb = plt.colorbar(o, extend='max', extendfrac = 'auto', shrink=0.25)
    cb_format = ticker.FuncFormatter(lambda x, pos: f'{x*1e9:.1f}')
    
    s_title = season.capitalize()
    cb.set_label('PM2.5 Concentration (μm)')
    cb.ax.yaxis.set_major_formatter(cb_format)
    plt.title(f'Mean {s_title} Tropospheric PM2.5 Concentration, {start} - {end}')

In [1]:
def map_seasonal_sfc_pm25(ds, season, time_range=None):
    # Handle dates
    if time_range != None:
        start = str(time_range[0])
        end = str(time_range[1])
        ds = ds.sel(time=slice(start, end))
    else:
        start = str((ds['time'].values)[0])[:4]
        end = str((ds['time'].values)[len(ds['time'])-1])[:4]

    # Handle seasons
    if str(season)=='winter':
         s = 'DJF'
    elif str(season)=='spring':
        s = 'MAM'
    elif str(season)=='summer':
        s = 'JJA'
    elif str(season)=='fall':
        s = 'SON'
    else:
        raise ValueError('Please choose a correct season from the list: ["winter", "spring", "summer", "fall"]')
            
    seasons = xr.groupers.SeasonResampler([s], drop_incomplete=False)
    ds_seasonal = ds.resample({'time':seasons}).mean() 

    try:
        layer_pm25 = ds_seasonal.PM25_SRF
    except:
        layer_pm25 = ds_seasonal.PM25.sel(lev=max(ds_seasonal.lev.values))

    # Convert from kg/m^3 to mg/m^3
    layer_pm25 = layer_pm25*1e9
    
    # Change longitude for plotting
    if max(ds_seasonal.lon)>350:
        layer_pm25 = layer_pm25.assign_coords(lon=(((layer_pm25.lon + 180) % 360) - 180)).sortby('lon')
    layer_pm25 = layer_pm25.mean(dim=['time'])
    layer_pm25 = layer_pm25

    # Create the plot
    proj = ccrs.PlateCarree()
    fig = plt.figure(figsize=(10,12))
    ax = plt.axes(projection = proj)
    ax.coastlines()
    ax.add_feature(cf.BORDERS.with_scale('50m'), edgecolor = [.3,.3,.3], linewidth = 0.5)

    data_cyc, lon_cyc = add_cyclic_point(layer_pm25, coord = layer_pm25.lon)
    o = ax.contourf(lon_cyc, layer_pm25.lat, data_cyc, transform=ccrs.PlateCarree(), cmap=matplotlib.cm.RdPu, levels=np.linspace(0,100,21), extend='max')
    cb = plt.colorbar(o, extend='max', extendfrac = 'auto', shrink=0.25)
    cb_format = ticker.FuncFormatter(lambda x, pos: f'{x*1:.1f}')
    
    s_title = season.capitalize()
    cb.set_label('PM2.5 Concentration (μg/m³)')
    cb.ax.yaxis.set_major_formatter(cb_format)
    plt.title(f'Mean {s_title} Surface PM2.5 Concentration, {start} - {end}')